# 🎬 Dublaj - AI Video Dubbing on Google Colab

## Multi-Speaker Voice Cloning with GPU Acceleration

### Features:
- ✅ FREE Tesla T4 GPU (15 GB VRAM)
- ✅ Automatic multi-speaker detection
- ✅ Voice cloning for each speaker
- ✅ 600+ languages supported
- ✅ Background music preservation

### Requirements:
- Google account
- Video file to dub
- API keys (see Setup section)

---

## 📋 Step 1: Enable GPU

**IMPORTANT:** Before running anything, enable GPU!

1. Click **Runtime** menu → **Change runtime type**
2. Select **Hardware accelerator**: **T4 GPU**
3. Click **Save**

Then verify GPU is enabled:

In [ ]:
# Verify GPU is available
!nvidia-smi

## 🔑 Step 2: Set API Keys

### Get Your API Keys:

1. **Gemini API Key** (Translation - REQUIRED):
   - Go to: https://aistudio.google.com/apikey
   - Click "Create API key"
   - Copy the key

2. **HuggingFace Token** (PyAnnote - REQUIRED):
   - Go to: https://huggingface.co/settings/tokens
   - Create new token (type: Fine-grained)
   - Accept licenses:
     - https://huggingface.co/pyannote/speaker-diarization-community-1
     - https://huggingface.co/pyannote/segmentation-3.0

### Enter Your Keys Below:

In [ ]:
import os
from google.colab import userdata

# Option 1: Use Colab Secrets (Recommended - more secure)
# Go to: 🔑 icon in left sidebar → Add secrets
# Add: GEMINI_API_KEY and HF_TOKEN
# Then uncomment these lines:
# GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
# HF_TOKEN = userdata.get('HF_TOKEN')

# Option 2: Direct entry (Less secure but easier)
GEMINI_API_KEY = "YOUR_GEMINI_KEY_HERE"  # Replace with your actual key
HF_TOKEN = "YOUR_HF_TOKEN_HERE"  # Replace with your actual token

# Set environment variables
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
os.environ['HF_TOKEN'] = HF_TOKEN

print("✅ API keys configured")

## 📦 Step 3: Install System Dependencies

In [ ]:
%%bash
# Install FFmpeg
apt-get update -qq
apt-get install -y -qq ffmpeg
echo "✅ FFmpeg installed: $(ffmpeg -version | head -n1)"

## 📥 Step 4: Clone Dublaj Repository

In [ ]:
# Clone the repository
!git clone https://github.com/Izahat/dublaj.git
%cd dublaj

# Install Python dependencies
!pip install -q -r requirements.txt

print("\n✅ Dublaj repository cloned and dependencies installed")

## 🐳 Step 5: Install Docker (This takes ~5 minutes)

In [ ]:
%%bash
# Install Docker
curl -fsSL https://get.docker.com -o get-docker.sh
sh get-docker.sh > /dev/null 2>&1

# Start Docker daemon in background
nohup dockerd > /tmp/dockerd.log 2>&1 &

# Wait for Docker to start
echo "Waiting for Docker to start..."
for i in {1..30}; do
  if docker ps > /dev/null 2>&1; then
    echo "✅ Docker is running"
    docker --version
    break
  fi
  sleep 2
done

## 🚀 Step 6: Start AI Model Services (Takes 10-15 minutes first time)

This downloads and starts all required AI models:
- Whisper (Speech Recognition) ~3 GB
- OmniVoice (TTS) ~5 GB
- Windowed RoFormer (Vocal Separation) ~2 GB
- PyAnnote (Speaker Diarization) ~3 GB
- Seed-VC (Voice Conversion) ~2 GB

**Note:** First run downloads models. Subsequent runs are instant.

In [ ]:
%%bash
# Start all Docker services
cd /content/dublaj

echo "Starting Whisper (Speech Recognition)..."
cd docker3/whisper
docker compose -f docker-compose.whisper.yml up -d

echo "\nStarting OmniVoice (TTS)..."
cd ../omnivoice
docker compose -f docker-compose.omnivoice.yml up -d --build

echo "\nStarting Windowed RoFormer (Vocal Separation)..."
cd ../windowed-roformer
docker compose -f docker-compose.windowed-roformer.yml up -d --build

echo "\nStarting PyAnnote (Speaker Diarization)..."
cd ../pyannote
docker compose -f docker-compose.pyannote.yml up -d --build

echo "\nStarting Seed-VC (Voice Conversion)..."
cd ../seedvc
docker compose -f docker-compose.seedvc.yml up -d --build

echo "\n✅ All services started!"
docker ps --format "table {{.Names}}\t{{.Status}}\t{{.Ports}}"

## ⏳ Step 7: Wait for Services to Initialize (5-10 minutes)

This monitors service health and waits until all are ready.

In [ ]:
import requests
import time
from IPython.display import clear_output

services = {
    "Whisper": "http://localhost:8100/health",
    "OmniVoice": "http://localhost:8200/health",
    "Windowed RoFormer": "http://localhost:8310/health",
    "PyAnnote": "http://localhost:8500/health",
    "Seed-VC": "http://localhost:8700/health",
}

def check_services():
    status = {}
    for name, url in services.items():
        try:
            response = requests.get(url, timeout=2)
            status[name] = "✅ Ready" if response.status_code == 200 else "⏳ Starting..."
        except:
            status[name] = "⏳ Starting..."
    return status

print("Waiting for all services to be ready...\n")
print("This may take 5-10 minutes on first run (downloading models)")
print("\nService Status:")

max_wait = 600  # 10 minutes
start_time = time.time()

while time.time() - start_time < max_wait:
    status = check_services()
    
    clear_output(wait=True)
    print("⏳ Waiting for services to initialize...\n")
    for name, state in status.items():
        print(f"{name:20s} {state}")
    
    if all("✅" in s for s in status.values()):
        print("\n🎉 All services are ready!")
        break
    
    time.sleep(10)
else:
    print("\n⚠️  Some services are still starting. You can proceed, but processing may fail.")
    print("Check Docker logs if issues occur.")

## 🎥 Step 8: Upload Your Video

Click the 📁 folder icon in the left sidebar → Upload your video file.

Or use the cell below to upload directly:

In [ ]:
from google.colab import files

print("📤 Upload your video file:")
uploaded = files.upload()

# Get the uploaded filename
video_filename = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {video_filename}")
print(f"File size: {len(uploaded[video_filename]) / 1024 / 1024:.2f} MB")

## 🌍 Step 9: Configure Dubbing Settings

In [ ]:
# Dubbing configuration
TARGET_LANGUAGE = "ru"  # Change to your target language

print("🎬 Dubbing Configuration:")
print(f"  Video: {video_filename}")
print(f"  Target Language: {TARGET_LANGUAGE}")
print("\n📝 Supported languages:")
print("  ru=Russian, en=English, tr=Turkish, az=Azerbaijani")
print("  es=Spanish, fr=French, de=German, it=Italian, pt=Portuguese")
print("  zh=Chinese, ja=Japanese, ko=Korean, ar=Arabic, hi=Hindi")
print("  Full list: http://localhost:8200/languages")

## 🚀 Step 10: Create .env Configuration

In [ ]:
%%writefile /content/dublaj/.env
# Dublaj Configuration for Google Colab

# General
APP_HOST=0.0.0.0
APP_PORT=8000
TEMP_DIR=./tmp
OUTPUT_DIR=./output
MAX_CONCURRENT_JOBS=1

# Whisper
WHISPER_BASE_URL=http://localhost:8100
WHISPER_ENDPOINT=/v1/audio/transcriptions

# Translation (using environment variables set earlier)
TRANSLATION_PROVIDER=gemini
TRANSLATION_MODEL=gemini-2.0-flash

# TTS
TTS_PROVIDER=omnivoice
OMNIVOICE_URL=http://localhost:8200
OMNIVOICE_VOICE=male

# Audio Separator
AUDIO_SEPARATOR_URL=http://localhost:8310
USE_VOICE_SEPARATION=true

# Seed-VC
SEEDVC_URL=http://localhost:8700
SEEDVC_MAX_PARALLEL=2

# PyAnnote
PYANNOTE_URL=http://localhost:8500

# FFmpeg
FFMPEG_PATH=ffmpeg

In [ ]:
# Add API keys to .env
with open('/content/dublaj/.env', 'a') as f:
    f.write(f"\nGEMINI_API_KEY={GEMINI_API_KEY}\n")
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

print("✅ Configuration file created")

## 🎬 Step 11: Process Video (Main Dubbing)

This runs the full dubbing pipeline:
1. Extract audio from video
2. Separate vocals from music
3. Detect speakers (PyAnnote)
4. Transcribe speech (Whisper)
5. Translate to target language
6. Synthesize speech (OmniVoice)
7. Apply voice conversion (Seed-VC)
8. Mix with background music
9. Merge into final video

**Processing time: ~5-15 minutes for a 5-minute video on T4 GPU**

In [ ]:
import subprocess
import json
from pathlib import Path

# Move uploaded video to dublaj directory
video_path = f"/content/dublaj/{video_filename}"
!mv "/content/{video_filename}" "{video_path}"

print("🎬 Starting dubbing process...\n")
print(f"Video: {video_filename}")
print(f"Target language: {TARGET_LANGUAGE}")
print("\nThis will take 5-15 minutes depending on video length.")
print("You can monitor progress below:\n")
print("="*60)

# Run dubbing via Python script
script = f"""
import asyncio
import sys
from pathlib import Path
sys.path.insert(0, '/content/dublaj')

from config.settings import get_settings
from providers.service_factory import ServiceFactory
from app.domain.entities import DubbingJob

async def main():
    settings = get_settings()
    factory = ServiceFactory(settings)
    orchestrator = factory.create_orchestrator()
    
    job = DubbingJob(target_language='{TARGET_LANGUAGE}')
    job.input_video_path = Path('{video_path}')
    
    result = await orchestrator.process_seedvc_v2_speaker_diarization_v2(job)
    
    if result.status.value == 'completed':
        print(f"\\n✅ SUCCESS! Output: {{result.output_video_path}}")
    else:
        print(f"\\n❌ FAILED: {{result.error_message}}")
    
    return result

result = asyncio.run(main())
"""

with open('/tmp/run_dubbing.py', 'w') as f:
    f.write(script)

!cd /content/dublaj && python /tmp/run_dubbing.py

## 📥 Step 12: Download Results

In [ ]:
from google.colab import files
import glob

# Find the output video
output_files = glob.glob('/content/dublaj/output/*/6_final_video.mp4')

if output_files:
    output_video = output_files[0]
    print(f"✅ Found output video: {output_video}")
    
    # Download the file
    print("\n📥 Downloading dubbed video...")
    files.download(output_video)
    
    # Show all artifacts
    job_dir = Path(output_video).parent
    print(f"\n📁 All output files in: {job_dir}")
    !ls -lh "{job_dir}"
    
    print("\n📝 You can also download:")
    print("  - 1_original_audio.wav")
    print("  - 2_clean_vocals.wav")
    print("  - 3_whisper_transcript.txt")
    print("  - 4_translation.txt")
    print("  - 5_synthesized_audio.wav")
    print("  - README.md")
else:
    print("❌ No output video found. Check logs above for errors.")

## 🔍 Troubleshooting

If something goes wrong, check these:

In [ ]:
# Check Docker services
print("📊 Docker Services Status:")
!docker ps

print("\n\n📝 Check service logs if needed:")
print("!docker logs dublaj-whisper")
print("!docker logs dublaj-omnivoice")
print("!docker logs dublaj-pyannote-diarization")
print("!docker logs dublaj-seedvc")

## 🧹 Cleanup (Optional)

Run this to stop services and free up memory:

In [ ]:
# Stop all Docker containers
!docker stop $(docker ps -q)

# Remove containers (keeps images for next run)
!docker rm $(docker ps -aq)

print("✅ Services stopped and cleaned up")

---

## 💡 Tips

1. **Processing time:**
   - 1-min video: ~3-5 minutes
   - 5-min video: ~10-20 minutes
   - 10-min video: ~20-40 minutes

2. **Colab session limits:**
   - Free tier: 12-hour session max
   - GPU usage: ~50 hours/week
   - If disconnected, models stay cached

3. **For multiple videos:**
   - Upload new video
   - Change TARGET_LANGUAGE if needed
   - Re-run Step 11 (services already running)

4. **Save work:**
   - Download output before closing
   - Files deleted when session ends

5. **Improve quality:**
   - Use shorter videos (<5 min)
   - Ensure clear audio in original
   - Check transcription accuracy

---

## 🎓 Learn More

- GitHub: https://github.com/Izahat/dublaj
- Documentation: See README.md in repository
- Issues: Report bugs on GitHub

---

**Happy Dubbing! 🎬🎙️**